# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents a single unique URL (web page) on a specific calendar date. The time window is the mid-panel month spanning strictly from March 1, 2026, to March 31, 2026. We are using this closed historical window to develop the features and train the model, intentionally leaving the final available month (June 2026) untouched as a sealed test set.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime

# Simulating the mid-panel month load from the Hugging Face warehouse
# In your actual notebook, use the HF datasets library to pull 'month=2026-03'
np.random.seed(42)
dates = pd.date_range(start="2026-03-01", end="2026-03-31")
urls = [f"https://example.com/page_{i}" for i in range(100)]

data = []
for d in dates:
    for u in urls:
        data.append({
            "date": d,
            "url": u,
            "impressions": np.random.randint(0, 1000),
            "clicks": np.random.randint(0, 50),
            "position": np.random.uniform(1.0, 50.0),
            "is_indexable": np.random.choice([True, False], p=[0.9, 0.1])
        })
df = pd.DataFrame(data)

print(f"Loaded {len(df)} rows.")
print(f"Date window strictly bounded: {df['date'].min().date()} to {df['date'].max().date()}")

Loaded 3100 rows.
Date window strictly bounded: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

- Feature: impressions_rolling_7d, avg_position_3d, is_weekend (strictly knowable at the decision moment).

- Label: target_clicks_gt_5 (binary: 1 if the URL receives > 5 clicks in the subsequent 7 days, 0 otherwise).

- Context: url, date, is_indexable (used for grouping, splitting, and filtering; never passed to the model).

- Excluded: current_day_clicks and future_7d_impressions are strictly excluded to prevent feature leakage. We also exclude any rows where is_indexable IS FALSE, because unindexed pages mathematically cannot generate search clicks, making ML predictions redundant.

In [2]:
# Defining the field buckets as a data dictionary mapping for the pipeline
features = ['impressions_rolling_7d', 'avg_position_3d', 'is_weekend']
label = 'target_clicks_gt_5'
context = ['url', 'date', 'is_indexable']
excluded = ['current_day_clicks', 'future_7d_impressions']

# Apply the exclusion rule immediately (Context filter)
df_model = df.query("is_indexable == True").copy()
print(f"Rows remaining after excluding unindexable context: {len(df_model)}")

Rows remaining after excluding unindexable context: 2801


## 3. Verify it with queries (grain, counts, missing values, windows)

1. The Grain: We check for duplicate (url, date) pairs to prove one row really is one URL per day.

2. Missing Values: We audit the core columns to ensure our features won't fail on nulls.

3. The Leakage Trap: We deliberately inject a label-derived column, watch the model score artificially spike, and then remove it to secure an honest baseline.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 1. Verify Grain (Should be 0)
grain_violations = df_model.duplicated(subset=['date', 'url']).sum()
print(f"Grain violations (duplicate url+date): {grain_violations}")

# 2. Missing Values Check
print("\nMissing values per column:")
print(df_model[['impressions', 'clicks', 'position']].isnull().sum())

# --- Feature Engineering ---
# Mocking the calculation of our features for the experiment
df_model['impressions_rolling_7d'] = df_model['impressions'] * np.random.uniform(0.8, 1.2, len(df_model))
df_model['avg_position_3d'] = df_model['position'] * np.random.uniform(0.9, 1.1, len(df_model))
df_model['is_weekend'] = df_model['date'].dt.dayofweek >= 5
df_model['target_clicks_gt_5'] = (np.random.rand(len(df_model)) > 0.7).astype(int)

# 3. The Leakage Trap Experiment
print("\n--- LEAKAGE EXPERIMENT ---")
# Deliberately leaking the target into a feature
df_model['LEAKED_future_clicks'] = df_model['target_clicks_gt_5'] * 10 + np.random.randint(0, 5, len(df_model))

X_leaked = df_model[features + ['LEAKED_future_clicks']]
y = df_model[label]

X_train, X_test, y_train, y_test = train_test_split(X_leaked, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
leaked_preds = clf.predict(X_test)
print(f"F1 Score WITH target leakage: {f1_score(y_test, leaked_preds):.3f} (Suspiciously high)")

# Remove the trap and get the honest baseline
X_honest = df_model[features]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
clf.fit(X_train_h, y_train_h)
honest_preds = clf.predict(X_test_h)
print(f"Honest F1 Score: {f1_score(y_test_h, honest_preds):.3f} (Ready for modeling)")

Grain violations (duplicate url+date): 0

Missing values per column:
impressions    0
clicks         0
position       0
dtype: int64

--- LEAKAGE EXPERIMENT ---
F1 Score WITH target leakage: 1.000 (Suspiciously high)
Honest F1 Score: 0.000 (Ready for modeling)


## 4. Data limits

This dataset is strictly bounded by Google Search Console reporting. It is blind to direct traffic, social media referrals, and email newsletter clicks. Consequently, a URL that receives zero search impressions might actually be a highly trafficked, valuable page driven by a viral social post. Our model can only predict search viability, not absolute content value. Furthermore, calculating rolling 7-day windows on the very first week of the dataset (March 1 - March 7) will result in truncated, unbalanced history since we lack the late-February context.

In [4]:
# Demonstrating the "Cold Start" boundary limitation
# Rows in the first 7 days have incomplete historical lookback windows
cutoff_date = pd.to_datetime("2026-03-07")
cold_start_rows = df_model[df_model['date'] <= cutoff_date]
steady_state_rows = df_model[df_model['date'] > cutoff_date]

print("Limitation: Unbalanced history in early rows")
print(f"Rows with incomplete 7-day lookback history (Mar 1-7): {len(cold_start_rows)}")
print(f"Rows with full historical context (Mar 8+): {len(steady_state_rows)}")
print("Note: In production, we must either drop the first 7 days or import late-February data to pad the rolling calculations.")

Limitation: Unbalanced history in early rows
Rows with incomplete 7-day lookback history (Mar 1-7): 621
Rows with full historical context (Mar 8+): 2180
Note: In production, we must either drop the first 7 days or import late-February data to pad the rolling calculations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.